In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv('https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv')

In [7]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [19]:
df = df[
    [
        'engine_displacement',
        'horsepower',
        'vehicle_weight',
        'model_year',
        'fuel_efficiency_mpg'
    ]
]

In [20]:
df.isnull().sum()

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
fuel_efficiency_mpg      0
dtype: int64

In [21]:
df["horsepower"].median()

np.float64(254.0)

In [22]:
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

In [23]:
def train_linear_regression(X, y):
    X = np.column_stack([np.ones(X.shape[0]), X])
    w = np.linalg.inv(X.T @ X) @ X.T @ y
    return w

def rmse(y, y_pred):
    return np.sqrt(np.mean((y - y_pred) ** 2))

In [24]:
mean = df_train.horsepower.mean()

for value in [0, mean]:
    train = df_train.copy()
    val = df_val.copy()

    train.horsepower = train.horsepower.fillna(value)
    val.horsepower = val.horsepower.fillna(value)

    X_train = train.drop('fuel_efficiency_mpg', axis=1).values
    y_train = train.fuel_efficiency_mpg.values

    X_val = val.drop('fuel_efficiency_mpg', axis=1).values
    y_val = val.fuel_efficiency_mpg.values

    w = train_linear_regression(X_train, y_train)
    y_pred = w[0] + X_val @ w[1:]

    print(round(rmse(y_val, y_pred), 3))

2.205
2.202


In [25]:
def train_linear_regression_reg(X, y, r):
    X = np.column_stack([np.ones(X.shape[0]), X])
    XTX = X.T @ X
    XTX = XTX + r * np.eye(XTX.shape[0])
    return np.linalg.solve(XTX, X.T @ y)

In [26]:
X_train = df_train.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
y_train = df_train.fuel_efficiency_mpg.values

X_val = df_val.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
y_val = df_val.fuel_efficiency_mpg.values

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    w = train_linear_regression_reg(X_train, y_train, r)
    y_pred = w[0] + X_val @ w[1:]
    print(r, round(rmse(y_val, y_pred), 4))

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [27]:
scores = []

for seed in range(10):
    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train = df.iloc[idx[:n_train]]
    df_val = df.iloc[idx[n_train:n_train + n_val]]

    X_train = df_train.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
    y_train = df_train.fuel_efficiency_mpg.values

    X_val = df_val.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
    y_val = df_val.fuel_efficiency_mpg.values

    w = train_linear_regression(X_train, y_train)
    y_pred = w[0] + X_val @ w[1:]

    scores.append(rmse(y_val, y_pred))

round(np.std(scores), 3)

np.float64(0.029)

In [28]:
np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]

df_train_full = pd.concat([df_train, df_val])

X_train = df_train_full.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
y_train = df_train_full.fuel_efficiency_mpg.values

X_test = df_test.drop('fuel_efficiency_mpg', axis=1).fillna(0).values
y_test = df_test.fuel_efficiency_mpg.values

w = train_linear_regression_reg(X_train, y_train, 0.001)

y_pred = w[0] + X_test @ w[1:]

round(rmse(y_test, y_pred), 3)

np.float64(2.236)